# Fine-tuned LoRA models: With CoT vs Without CoT (closed Yes/No questions) -- Kaggle

Evaluates every fine-tuned model `Arup330/{Dataset}_closed_{withCoT|noCoT}_{Model}_lora` for BrainFace, Neck and Abdomen in **one run**.

**Before running**
1. Settings -> Accelerator -> GPU, and Internet -> On.
2. Add the dataset `arups330/slackdataset` to the notebook (Add Input).
3. Add-ons -> Secrets -> add `HF_TOKEN` and attach it to this notebook.
4. Optional quick test: set `MAX_ROWS_PER_DATASET = 5` in section 1, Run All, confirm all 30 evaluations finish without errors. Then set it back to `None` and Run All again for the real results.

Results are written to `/kaggle/working/cot_eval_results` (Output panel).

Each run loads the matching `Arup330/...` repository directly with
`FastVisionModel.from_pretrained()`. Both training variants use the same `systemPrompt(question)`.
Test CSVs require question, answer, and image filename fields only.
The original CT data paths, prediction parsing, metrics, and summary tables are
preserved. Fresh predictions go into `raw_hf_models` to avoid reusing earlier
evaluations. The repositories store LoRA checkpoints; Unsloth resolves
their recorded base-model dependencies automatically with PEFT installed.


Kaggle: choose T4 x2, enable Internet, and run one model at a time in 4-bit
on GPU 0 with FP16 compute. The second GPU is not pooled automatically.
First smoke-test with MAX_ROWS_PER_DATASET=1; then reset to None for full evaluation.
All three datasets and all 30 models are required: errors stop execution rather
than silently dropping a model or image. Completed model results are saved.
If splitting sessions, save/download the raw_hf_models folder INCLUDING metadata,
attach that saved output as a Kaggle input, and set RESUME_RAW_DIRS to its path.
Set RUN_DATASETS/RUN_MODELS for the next batch. Keep the same evaluation settings.
The final section refuses partial or smoke-test reports and prints all 30 rows,
three 10-row tables, and the comparison table; it also writes FINAL_MODEL_SUMMARY.txt.


Small-test-set optimization: decoded images are cached in a bounded 256 MiB CPU
cache, so repeated image paths avoid repeated disk reads and decoding. Every
question still receives a separate prediction and model-specific vision processing.
Only one model and one question are processed on GPU at a time. All six runs for
each model family are adjacent, and the Hugging Face disk cache is kept for reuse.
The notebook prints question/image counts and saves measured load/inference times
beside each run's predictions. Image resolution and the shared prompt are unchanged.


In [1]:
%pip install -q -U unsloth scikit-learn pillow


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.3/81.3 kB 1.6 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 26.1/26.1 MB 49.7 MB/s eta 0:00:0000:0100:01m
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 66.6 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 44.7 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 18.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.8/73.8 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 68.8 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 17.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 51.8 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 52.5 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 45.8 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 216.9/216.9 kB 9.

In [11]:
import os, re, gc, json, glob, time, shutil, hashlib
from collections import OrderedDict
from unsloth import FastVisionModel
import torch
import pandas as pd
from PIL import Image
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from huggingface_hub import login

# ---------------------------------------------------------------------------
# Hugging Face token. Needed if your LoRA repos are private or a base model is
# gated (e.g. google/medgemma-4b-it).
# In Kaggle: Add-ons -> Secrets -> add HF_TOKEN -> attach it to this notebook.
# ---------------------------------------------------------------------------
HF_TOKEN = os.environ.get("HF_TOKEN") or os.environ.get("HF_Token")
try:
    from kaggle_secrets import UserSecretsClient
    for secret_name in ("HF_TOKEN", "HF_Token"):
        try:
            HF_TOKEN = UserSecretsClient().get_secret(secret_name) or HF_TOKEN
            if HF_TOKEN:
                break
        except Exception:
            continue
except Exception:
    pass

if HF_TOKEN:
    login(token=HF_TOKEN)
    print("Logged in to Hugging Face.")
else:
    print("No HF_TOKEN found -- loading a private or gated dependency may fail.")

Logged in to Hugging Face.


## 1. Configuration

In [12]:
# ---------------------------------------------------------------------------
# 1. Configuration -- everything you may need to change is in this cell
# ---------------------------------------------------------------------------
HF_USER = "Arup330"

# Folder that contains the dataset folders (Abdomen/, Neck/, BrainFace/ ...)
DATA_ROOT = "/kaggle/input/datasets/arups330/slackdataset/SLAKE_split_en_realimages"

# Name used in the Hugging Face repo names -> folder name inside DATA_ROOT.
# The test set is read from <DATA_ROOT>/<folder>/<modality>/test/closed.csv
DATASETS = {
    "BrainFace": "Brain_Face",
    "Neck":      "Neck",
    "Abdomen":   "Abdomen",
}

# Model suffixes used by your Hugging Face repositories.
MODELS = [
    "Lingshu-7B",
    "MedGemma-4B",
    "Qwen2.5-VL-7B",
    "Qwen3-VL-8B",
    "Llama-11B",
]

# (label in the summary table, tag in the repo name)
CONDITIONS = [("With", "withCoT"), ("Without", "noCoT")]

def model_repo(dataset: str, tag: str, model_name: str) -> str:
    # e.g. Arup330/BrainFace_closed_withCoT_Lingshu-7B_lora
    return f"{HF_USER}/{dataset}_closed_{tag}_{model_name}_lora"

# All 30 are required for the final report. Subsets allow work across sessions.
RUN_DATASETS = list(DATASETS)  # e.g. ["BrainFace"] for one session
RUN_MODELS = list(MODELS)     # e.g. ["Llama-11B"] to resume one family
# Optional previous Kaggle output folders containing raw_hf_models:
RESUME_RAW_DIRS = []  # e.g. ["/kaggle/input/previous-results/cot_eval_results/raw_hf_models"]
assert set(RUN_DATASETS) <= set(DATASETS)
assert set(RUN_MODELS) <= set(MODELS)

MAX_ROWS_PER_DATASET = None   # e.g. 5 for a quick smoke test of all 30 runs
MAX_IMAGE_SIDE = None         # e.g. 1024 to downscale very large images (faster, less VRAM)
MAX_NEW_TOKENS = 8
IMAGE_CACHE_MB = 256  # Decoded RGB images in CPU RAM, not GPU memory.

# What to do when a model answers something other than Yes/No:
#   "wrong"  -> counted as a wrong answer (shown in the "Invalid" column)
#   "ignore" -> dropped from that model's metrics
INVALID_PRED_POLICY = "wrong"


# Every finished run is saved here (shown in the notebook's Output panel).
# Re-running the notebook in the same session continues where it stopped.
RESULTS_DIR = "/kaggle/working/cot_eval_results"
RAW_DIR = os.path.join(RESULTS_DIR, "raw_hf_models")
os.makedirs(RAW_DIR, exist_ok=True)

## 2. Test data

In [13]:
# ---------------------------------------------------------------------------
# 2. Load the closed-question TEST set of every dataset (Yes/No rows only)
#    Reads <DATA_ROOT>/<folder>/<modality>/test/closed.csv directly -- no zip.
# ---------------------------------------------------------------------------
def gold_yesno(text) -> str:
    """Gold answer must be exactly Yes or No, otherwise the row is ignored."""
    m = re.fullmatch(r"\W*(yes|no)\W*", str(text).strip().lower())
    return m.group(1) if m else "invalid"

def pred_yesno(text) -> str:
    """Model output must start with the word Yes or No ('Yes.', '**No**' are fine;
    'Normal', 'Not seen', 'The answer is yes' are invalid)."""
    m = re.match(r"[^a-z]*(yes|no)\b", str(text).strip().lower())
    return m.group(1) if m else "invalid"

def resolve_image_path(split_dir: str, img_name: str):
    img_name = str(img_name)
    for c in [os.path.join(split_dir, img_name),
              os.path.join(split_dir, os.path.basename(img_name)),
              os.path.join(split_dir, img_name.replace("/", "_"))]:
        if os.path.exists(c):
            return c
    return None

# Bound CPU memory while reusing repeated images across questions/models.
IMAGE_CACHE = OrderedDict()
IMAGE_CACHE_BYTES = 0

def load_image(path: str) -> Image.Image:
    global IMAGE_CACHE_BYTES
    key = (os.path.abspath(path), MAX_IMAGE_SIDE)
    if key in IMAGE_CACHE:
        IMAGE_CACHE.move_to_end(key)
        return IMAGE_CACHE[key].copy()
    with Image.open(path) as source:
        img = source.convert("RGB")
    if MAX_IMAGE_SIDE and max(img.size) > MAX_IMAGE_SIDE:
        img.thumbnail((MAX_IMAGE_SIDE, MAX_IMAGE_SIDE), Image.LANCZOS)
    size_bytes = img.width * img.height * 3
    budget = int(IMAGE_CACHE_MB * 1024 * 1024)
    if size_bytes <= budget:
        while IMAGE_CACHE and IMAGE_CACHE_BYTES + size_bytes > budget:
            _, old = IMAGE_CACHE.popitem(last=False)
            IMAGE_CACHE_BYTES -= old.width * old.height * 3
            old.close()
        IMAGE_CACHE[key] = img
        IMAGE_CACHE_BYTES += size_bytes
        return img.copy()  # Caller can close/mutate this safely.
    return img

def load_closed_test(name: str, folder: str) -> pd.DataFrame:
    # only the test split: .../<folder>/CT/test/closed.csv, .../<folder>/MRI/test/closed.csv, ...
    csvs = sorted(glob.glob(os.path.join(DATA_ROOT, folder, "CT", "test", "closed.csv"), recursive=True))
    if not csvs:
        raise FileNotFoundError(f"no test/closed.csv found under {os.path.join(DATA_ROOT, folder)}")

    frames = []
    for csv_path in csvs:
        df = pd.read_csv(csv_path)
        df["split_dir"] = os.path.dirname(csv_path)
        frames.append(df)
        print(f"  {csv_path}  ({len(df)} rows)")
    df = pd.concat(frames, ignore_index=True)
    n_total = len(df)

    # keep only rows whose gold answer is Yes/No
    df["gold"] = df["answer"].map(gold_yesno)
    df = df[df["gold"] != "invalid"]
    n_yesno = len(df)

    img_col = "image_file" if "image_file" in df.columns else "img_name"
    df["img_path"] = [resolve_image_path(d, n) for d, n in zip(df["split_dir"], df[img_col])]
    missing_images = df.loc[df["img_path"].isna(), img_col].tolist()
    if missing_images:
        raise FileNotFoundError(
            f"{name}: {len(missing_images)} test images are missing: "
            f"{missing_images[:5]}. Fix the image paths before evaluation."
        )

    if MAX_ROWS_PER_DATASET:
        df = df.head(MAX_ROWS_PER_DATASET)
    df = df.reset_index(drop=True)

    if df.empty:
        raise ValueError(f"{name}: no usable Yes/No test rows")
    counts = df["gold"].value_counts().to_dict()
    print(f"{name}: {n_total} rows -> {n_yesno} with Yes/No answer "
          f"-> {len(df)} used  (yes={counts.get('yes', 0)}, no={counts.get('no', 0)})\n")
    return df[["question", "gold", "img_path"]]

TEST = {}            # dataset -> DataFrame
for name, folder in DATASETS.items():
    TEST[name] = load_closed_test(name, folder)

if not TEST and os.path.isdir(DATA_ROOT):
    print("Folders found in DATA_ROOT:", os.listdir(DATA_ROOT))
assert TEST, "No dataset loaded -- check DATA_ROOT and the folder names in section 1."
print(f"Datasets ready: {list(TEST)} | {sum(len(d) for d in TEST.values())} questions")
print("\nEvaluation workload (every question is retained):")
for name, df in TEST.items():
    unique_images = df["img_path"].nunique()
    print(f"  {name}: {len(df)} questions, {unique_images} unique image paths, "
          f"{len(MODELS) * len(CONDITIONS) * len(df)} model predictions")
print(f"Total predictions: {len(MODELS) * len(CONDITIONS) * sum(map(len, TEST.values()))}")


  /kaggle/input/datasets/arups330/slackdataset/SLAKE_split_en_realimages/Brain_Face/CT/test/closed.csv  (12 rows)
BrainFace: 12 rows -> 12 with Yes/No answer -> 12 used  (yes=5, no=7)

  /kaggle/input/datasets/arups330/slackdataset/SLAKE_split_en_realimages/Neck/CT/test/closed.csv  (16 rows)
Neck: 16 rows -> 16 with Yes/No answer -> 16 used  (yes=7, no=9)

  /kaggle/input/datasets/arups330/slackdataset/SLAKE_split_en_realimages/Abdomen/CT/test/closed.csv  (127 rows)
Abdomen: 127 rows -> 108 with Yes/No answer -> 108 used  (yes=50, no=58)

Datasets ready: ['BrainFace', 'Neck', 'Abdomen'] | 136 questions

Evaluation workload (every question is retained):
  BrainFace: 12 questions, 2 unique image paths, 120 model predictions
  Neck: 16 questions, 3 unique image paths, 160 model predictions
  Abdomen: 108 questions, 22 unique image paths, 1080 model predictions
Total predictions: 1360


## 3. Prompts

In [14]:
def systemPrompt(question: str) -> str:
    return f"""Context:
- You are a medical visual question answering (MedVQA) expert specializing in the interpretation of medical images.

Objective:
- Answer the closed-ended question based on the visual evidence in the provided medical image.

Inputs:
- Question: {question}

Instructions:
1. Examine the medical image carefully.
2. Identify the visual findings relevant to the question.
3. Interpret the question precisely, including any negation.
4. Select the answer best supported by the image.
5. Do not fabricate findings or make unsupported assumptions.

Output Requirements:
- Output exactly one word: Yes or No.
- Do not provide reasoning, explanations, punctuation, or additional text.

Valid outputs:
Yes
No
"""


## 4. Model helpers

In [15]:
from sklearn.metrics import classification_report, accuracy_score


def load_model(repo: str):
    if not torch.cuda.is_available():
        raise RuntimeError(
            "Enable a CUDA GPU before running model evaluation."
        )

    model, tokenizer = FastVisionModel.from_pretrained(
        repo,
        load_in_4bit=True,
        dtype=torch.float16,
        device_map={"": "cuda:0"},
        use_gradient_checkpointing="unsloth",
        token=HF_TOKEN,
    )

    FastVisionModel.for_inference(model)
    return model, tokenizer


@torch.inference_mode()
def generate_answer(model, tokenizer, image, prompt_text: str) -> str:
    messages = [{
        "role": "user",
        "content": [
            {"type": "image"},
            {"type": "text", "text": prompt_text},
        ],
    }]

    input_text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    inputs = tokenizer(
        images=image,
        text=input_text,
        add_special_tokens=False,
        return_tensors="pt",
    ).to(model.device)

    output_ids = model.generate(
        **inputs,
        max_new_tokens=MAX_NEW_TOKENS,
        do_sample=False,
        use_cache=True,
    )

    generated = output_ids[0][inputs["input_ids"].shape[1]:]

    return tokenizer.decode(
        generated,
        skip_special_tokens=True,
    ).strip()


def run_condition(model, tokenizer, df: pd.DataFrame, tag: str):
    results = []

    for i, row in enumerate(df.itertuples(index=False)):
        prompt = systemPrompt(row.question)
        image = load_image(row.img_path)

        try:
            raw = generate_answer(
                model,
                tokenizer,
                image,
                prompt,
            )
        except Exception as e:
            raise RuntimeError(
                f"{tag}: test row {i} failed: {e}"
            ) from e
        finally:
            image.close()

        pred = pred_yesno(raw)

        results.append({
            "row": i,
            "image": os.path.basename(row.img_path),
            "question": row.question,
            "gold": row.gold,
            "pred": pred,
            "raw": raw,
            "error": None,
        })

        print(
            f"    [{i}/{len(df)}] "
            f"gold={row.gold} pred={pred} raw={raw!r}",
            flush=True,
        )

    return results


def print_model_report(results, model_name, condition, dataset):
    y_true, y_pred = [], []
    n_invalid = 0

    for r in results:
        if r["pred"] in ("yes", "no"):
            y_true.append(r["gold"])
            y_pred.append(r["pred"])
        else:
            n_invalid += 1

            # Preserve the existing final-summary scoring policy.
            if INVALID_PRED_POLICY == "wrong":
                y_true.append(r["gold"])
                y_pred.append(
                    "no" if r["gold"] == "yes" else "yes"
                )

    print(
        f"\nClassification report -- {model_name} "
        f"({condition} CoT, {dataset}):",
        flush=True,
    )

    if not y_true:
        print("No predictions available for scoring.")
        return

    print(
        classification_report(
            y_true,
            y_pred,
            labels=["no", "yes"],
            target_names=["no", "yes"],
            digits=2,
            zero_division=0,
        ),
        flush=True,
    )

    accuracy = accuracy_score(y_true, y_pred) * 100

    print(f"Accuracy: {accuracy:.2f}%", flush=True)
    print(
        f"Invalid predictions: {n_invalid}/{len(results)}\n",
        flush=True,
    )

## 5. Run all models

In [16]:
def result_path(dataset, model_name, condition):
    return os.path.join(
        RAW_DIR,
        f"{dataset}__{model_name}__{condition}.json",
    )


IMAGE_HASHES = {}


def image_hash(path):
    if path not in IMAGE_HASHES:
        digest = hashlib.sha256()

        with open(path, "rb") as f:
            for chunk in iter(lambda: f.read(1024 * 1024), b""):
                digest.update(chunk)

        IMAGE_HASHES[path] = digest.hexdigest()

    return IMAGE_HASHES[path]


def run_signature(dataset, model_name, condition):
    rows = [
        {
            "question": str(row.question),
            "gold": row.gold,
            "image_sha256": image_hash(row.img_path),
        }
        for row in TEST[dataset].itertuples(index=False)
    ]

    payload = {
        "repo": model_repo(
            dataset,
            dict(CONDITIONS)[condition],
            model_name,
        ),
        "rows": rows,
        "tokens": MAX_NEW_TOKENS,
        "image_side": MAX_IMAGE_SIDE,
        "prompt": systemPrompt("example"),
        "version": 6,
    }

    return hashlib.sha256(
        json.dumps(payload, sort_keys=True).encode()
    ).hexdigest()


def valid_result_file(path, dataset, model_name, condition):
    try:
        with open(path) as f:
            rows = json.load(f)

        with open(path + ".meta.json") as f:
            meta = json.load(f)

        return (
            len(rows) == len(TEST[dataset])
            and not any(r.get("error") for r in rows)
            and meta["signature"]
            == run_signature(dataset, model_name, condition)
        )

    except (OSError, ValueError, KeyError, TypeError):
        return False


def is_done(dataset, model_name, condition):
    return valid_result_file(
        result_path(dataset, model_name, condition),
        dataset,
        model_name,
        condition,
    )


def restore_results():
    for dataset in DATASETS:
        for model_name in MODELS:
            for condition, _ in CONDITIONS:
                if is_done(dataset, model_name, condition):
                    continue

                target = result_path(
                    dataset, model_name, condition
                )

                for directory in RESUME_RAW_DIRS:
                    source = os.path.join(
                        directory,
                        os.path.basename(target),
                    )

                    if valid_result_file(
                        source, dataset, model_name, condition
                    ):
                        shutil.copy2(source, target)
                        shutil.copy2(
                            source + ".meta.json",
                            target + ".meta.json",
                        )

                        print(
                            f"[RESTORED] {dataset} | "
                            f"{model_name} | {condition}"
                        )
                        break


def save_results(dataset, model_name, condition, results):
    path = result_path(dataset, model_name, condition)

    with open(path + ".tmp", "w") as f:
        json.dump(results, f, indent=2)

    os.replace(path + ".tmp", path)

    with open(path + ".meta.json", "w") as f:
        json.dump(
            {
                "signature": run_signature(
                    dataset, model_name, condition
                ),
                "model_repo": model_repo(
                    dataset,
                    dict(CONDITIONS)[condition],
                    model_name,
                ),
            },
            f,
            indent=2,
        )


restore_results()

run_start = time.time()
RUN_TIMINGS = []

for model_name in RUN_MODELS:
    for dataset in RUN_DATASETS:
        for condition, tag in CONDITIONS:
            repo = model_repo(dataset, tag, model_name)

            print(
                f"\n--- {model_name} | CoT: {condition} "
                f"| Dataset: {dataset} ---",
                flush=True,
            )

            # Print reports even when reusing completed results.
            if is_done(dataset, model_name, condition):
                print(
                    f"[DONE] Reusing verified results: {repo}",
                    flush=True,
                )

                with open(
                    result_path(dataset, model_name, condition)
                ) as f:
                    results = json.load(f)

                for r in results:
                    print(
                        f"    [{r['row']}/{len(results)}] "
                        f"gold={r['gold']} pred={r['pred']} "
                        f"raw={r['raw']!r}",
                        flush=True,
                    )

                print_model_report(
                    results,
                    model_name,
                    condition,
                    dataset,
                )
                continue

            model = tokenizer = None

            try:
                print(f"Loading: {repo}", flush=True)

                load_start = time.time()
                model, tokenizer = load_model(repo)
                load_seconds = time.time() - load_start

                inference_start = time.time()

                results = run_condition(
                    model,
                    tokenizer,
                    TEST[dataset],
                    tag=(
                        f"{dataset} | {model_name} | "
                        f"{condition} CoT"
                    ),
                )

                inference_seconds = (
                    time.time() - inference_start
                )

                save_results(
                    dataset,
                    model_name,
                    condition,
                    results,
                )

                # Print immediately after this model finishes.
                print_model_report(
                    results,
                    model_name,
                    condition,
                    dataset,
                )

                timing = {
                    "Dataset": dataset,
                    "Model Name": model_name,
                    "CoT": condition,
                    "Questions": len(results),
                    "Load seconds": round(load_seconds, 2),
                    "Inference seconds": round(
                        inference_seconds, 2
                    ),
                    "Seconds/question": round(
                        inference_seconds / len(results), 3
                    ),
                }

                RUN_TIMINGS.append(timing)

                with open(
                    result_path(dataset, model_name, condition)
                    + ".timing.json",
                    "w",
                ) as f:
                    json.dump(timing, f, indent=2)

                print(
                    f"Load: {load_seconds:.1f}s | "
                    f"Inference: {inference_seconds:.1f}s | "
                    f"Per question: "
                    f"{inference_seconds / len(results):.2f}s",
                    flush=True,
                )

            except Exception as e:
                raise RuntimeError(
                    f"Evaluation stopped at {repo}. "
                    "Completed results are saved. "
                    "Fix this error and rerun to resume: "
                    f"{type(e).__name__}: {e}"
                ) from e

            finally:
                model = tokenizer = None
                gc.collect()
                torch.cuda.empty_cache()

print(
    f"\nSelected runs finished in "
    f"{(time.time() - run_start) / 60:.1f} min",
    flush=True,
)


--- Lingshu-7B | CoT: With | Dataset: BrainFace ---
Loading: Arup330/BrainFace_closed_withCoT_Lingshu-7B_lora
==((====))==  Unsloth 2026.9.14: Fast Qwen2_5_Vl patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.562 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/729 [00:00<?, ?it/s]

Unsloth: Offloading embeddings to RAM to save 1.02 GB.


/usr/local/lib/python3.13/dist-packages/huggingface_hub/constants.py:299: FutureWarning: The `HF_HUB_ENABLE_HF_TRANSFER` environment variable is deprecated as 'hf_transfer' is not used anymore. Please use `HF_XET_HIGH_PERFORMANCE` instead to enable high performance transfer with Xet. Visit https://huggingface.co/docs/huggingface_hub/package_reference/environment_variables#hfxethighperformance for more details.
  warnings.warn(


    [0/12] gold=yes pred=yes raw='Yes'
    [1/12] gold=no pred=no raw='No'
    [2/12] gold=yes pred=yes raw='Yes'
    [3/12] gold=no pred=no raw='No.'
    [4/12] gold=no pred=no raw='No'
    [5/12] gold=no pred=no raw='No'
    [6/12] gold=yes pred=yes raw='Yes'
    [7/12] gold=no pred=no raw='No.'
    [8/12] gold=yes pred=yes raw='Yes'
    [9/12] gold=yes pred=yes raw='Yes'
    [10/12] gold=no pred=no raw='No.'
    [11/12] gold=no pred=no raw='No'

Classification report -- Lingshu-7B (With CoT, BrainFace):
              precision    recall  f1-score   support

          no       1.00      1.00      1.00         7
         yes       1.00      1.00      1.00         5

    accuracy                           1.00        12
   macro avg       1.00      1.00      1.00        12
weighted avg       1.00      1.00      1.00        12

Accuracy: 100.00%
Invalid predictions: 0/12

Load: 91.4s | Inference: 18.4s | Per question: 1.54s

--- Lingshu-7B | CoT: Without | Dataset: BrainFace ---
Loading

...

/usr/local/lib/python3.13/dist-packages/huggingface_hub/constants.py:299: FutureWarning: The `HF_HUB_ENABLE_HF_TRANSFER` environment variable is deprecated as 'hf_transfer' is not used anymore. Please use `HF_XET_HIGH_PERFORMANCE` instead to enable high performance transfer with Xet. Visit https://huggingface.co/docs/huggingface_hub/package_reference/environment_variables#hfxethighperformance for more details.
  warnings.warn(


    [0/108] gold=no pred=no raw='No'
    [1/108] gold=yes pred=yes raw='Yes'
    [2/108] gold=no pred=no raw='No'
    [3/108] gold=yes pred=no raw='No'
    [4/108] gold=yes pred=yes raw='Yes'
    [5/108] gold=no pred=no raw='No'
    [6/108] gold=no pred=no raw='No'
    [7/108] gold=no pred=no raw='No'
    [8/108] gold=yes pred=no raw='No'
    [9/108] gold=yes pred=no raw='No'
    [10/108] gold=yes pred=no raw='No'
    [11/108] gold=no pred=no raw='No'
    [12/108] gold=no pred=no raw='No'
    [13/108] gold=yes pred=no raw='No'
    [14/108] gold=yes pred=yes raw='Yes'
    [15/108] gold=yes pred=no raw='No'
    [16/108] gold=yes pred=yes raw='Yes'
    [17/108] gold=no pred=yes raw='Yes'
    [18/108] gold=yes pred=yes raw='Yes'
    [19/108] gold=yes pred=no raw='No'
    [20/108] gold=no pred=no raw='No'
    [21/108] gold=yes pred=yes raw='Yes'
    [22/108] gold=no pred=yes raw='Yes'
    [23/108] gold=yes pred=no raw='No'
    [24/108] gold=no pred=no raw='No'
    [25/108] gold=no pred=no r

Loading weights:   0%|          | 0/906 [00:00<?, ?it/s]

/usr/local/lib/python3.13/dist-packages/huggingface_hub/constants.py:299: FutureWarning: The `HF_HUB_ENABLE_HF_TRANSFER` environment variable is deprecated as 'hf_transfer' is not used anymore. Please use `HF_XET_HIGH_PERFORMANCE` instead to enable high performance transfer with Xet. Visit https://huggingface.co/docs/huggingface_hub/package_reference/environment_variables#hfxethighperformance for more details.
  warnings.warn(


    [0/108] gold=no pred=no raw='No'
    [1/108] gold=yes pred=yes raw='Yes'
    [2/108] gold=no pred=no raw='No'
    [3/108] gold=yes pred=yes raw='Yes'
    [4/108] gold=yes pred=yes raw='Yes'
    [5/108] gold=no pred=yes raw='Yes'
    [6/108] gold=no pred=yes raw='Yes'
    [7/108] gold=no pred=yes raw='Yes'
    [8/108] gold=yes pred=yes raw='Yes'
    [9/108] gold=yes pred=yes raw='Yes'
    [10/108] gold=yes pred=yes raw='Yes'
    [11/108] gold=no pred=yes raw='Yes'
    [12/108] gold=no pred=yes raw='Yes'
    [13/108] gold=yes pred=yes raw='Yes'
    [14/108] gold=yes pred=yes raw='Yes'
    [15/108] gold=yes pred=yes raw='Yes'
    [16/108] gold=yes pred=yes raw='Yes'
    [17/108] gold=no pred=yes raw='Yes'
    [18/108] gold=yes pred=yes raw='Yes'
    [19/108] gold=yes pred=yes raw='Yes'
    [20/108] gold=no pred=yes raw='Yes'
    [21/108] gold=yes pred=yes raw='Yes'
    [22/108] gold=no pred=yes raw='Yes'
    [23/108] gold=yes pred=yes raw='Yes'
    [24/108] gold=no pred=yes raw='Yes'


## 6. Summary tables

In [17]:
# ---------------------------------------------------------------------------
# 6. Summary tables (built from the saved results, so this cell can be re-run
#    at any time, even after a restart)
# ---------------------------------------------------------------------------
def compute_metrics(results):
    n_invalid = sum(r["pred"] not in ("yes", "no") for r in results)
    y_true, y_pred = [], []
    for r in results:
        if r["pred"] in ("yes", "no"):
            y_true.append(r["gold"]); y_pred.append(r["pred"])
        elif INVALID_PRED_POLICY == "wrong":
            y_true.append(r["gold"]); y_pred.append("no" if r["gold"] == "yes" else "yes")
    if not y_true:
        return None

    labels = ["no", "yes"]
    acc = accuracy_score(y_true, y_pred) * 100
    mp, mr, mf, _ = precision_recall_fscore_support(y_true, y_pred, labels=labels, average="macro", zero_division=0)
    wp, wr, wf, _ = precision_recall_fscore_support(y_true, y_pred, labels=labels, average="weighted", zero_division=0)
    cp, cr, cf, _ = precision_recall_fscore_support(y_true, y_pred, labels=labels, zero_division=0)
    fmt = lambda k: f"{cp[k] * 100:.1f}/{cr[k] * 100:.1f}/{cf[k] * 100:.1f}"
    return {
        "N": len(y_true),
        "Invalid": n_invalid,
        "Accuracy (%)": round(acc, 2),
        "Precision (macro)": round(mp * 100, 2),
        "Recall (macro)": round(mr * 100, 2),
        "F1 (macro)": round(mf * 100, 2),
        "Precision (weighted)": round(wp * 100, 2),
        "Recall (weighted)": round(wr * 100, 2),
        "F1 (weighted)": round(wf * 100, 2),
        "Yes-class P/R/F1": fmt(1),
        "No-class P/R/F1": fmt(0),
    }

# Publish the report only after all 30 current evaluations are complete.
required_runs = [(d, m, c) for d in DATASETS for m in MODELS for c, _ in CONDITIONS]
unfinished = [f"{d} | {m} | {c}" for d, m, c in required_runs if not is_done(d, m, c)]
if unfinished:
    raise RuntimeError(
        f"FINAL SUMMARY NOT READY: {len(required_runs) - len(unfinished)}/"
        f"{len(required_runs)} completed. Resume these runs:\n" + "\n".join(unfinished)
    )
if MAX_ROWS_PER_DATASET is not None:
    raise RuntimeError("Smoke test complete. Set MAX_ROWS_PER_DATASET=None and rerun all models for the final report.")

summary_rows, missing = [], []
for dataset in DATASETS:
    raw_all = {}
    for model_name in MODELS:
        for condition, _ in CONDITIONS:
            p = result_path(dataset, model_name, condition)
            if not os.path.exists(p):
                if dataset in TEST:
                    missing.append(f"{dataset} | {model_name} | {condition}")
                continue
            with open(p) as f:
                results = json.load(f)
            raw_all[f"{model_name}_{condition}"] = results
            metrics = compute_metrics(results)
            if metrics:
                summary_rows.append({"Dataset": dataset, "Model Name": model_name, "CoT": condition, **metrics})
    if raw_all:
        with open(os.path.join(RESULTS_DIR, f"cot_comparison_raw_results_{dataset}.json"), "w") as f:
            json.dump(raw_all, f, indent=2)

summary_df = pd.DataFrame(summary_rows)
assert len(summary_df), "No results yet -- run section 5 first."

pd.set_option("display.width", 300)
pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)

print("\nFINAL REPORT: 30/30 MODELS COMPLETED — FULL TEST SETS")
print("Input: image + shared question prompt | Output: Yes/No | CoT labels: training variant")
print(f"Precision: 4-bit weights / FP16 compute | Max new tokens: {MAX_NEW_TOKENS}")
print(f"Image size cap: {MAX_IMAGE_SIDE} | Invalid prediction policy: {INVALID_PRED_POLICY}")
print("\nALL DATASETS — ALL 30 MODELS — ALL METRICS")
print(summary_df.to_string(index=False))

for dataset in summary_df["Dataset"].unique():
    part = summary_df[summary_df["Dataset"] == dataset]
    print("\n" + "=" * 100)
    print(f"FULL SUMMARY TABLE -- {dataset} -- fine-tuned models, With vs Without CoT")
    print("=" * 100)
    print(part.to_string(index=False))
    part.to_csv(os.path.join(RESULTS_DIR, f"cot_comparison_summary_{dataset}.csv"), index=False)

summary_df.to_csv(os.path.join(RESULTS_DIR, "cot_comparison_summary_ALL.csv"), index=False)

# Compact side-by-side view: With vs Without CoT
compact = summary_df.pivot_table(index=["Dataset", "Model Name"], columns="CoT",
                                 values=["Accuracy (%)", "F1 (macro)"], sort=False)
for metric in ["Accuracy (%)", "F1 (macro)"]:
    if {"With", "Without"} <= set(compact[metric].columns):
        compact[(metric, "Diff")] = (compact[(metric, "With")] - compact[(metric, "Without")]).round(2)
compact = compact.sort_index(axis=1, level=0, sort_remaining=False)
print("\n" + "=" * 100)
print("WITH vs WITHOUT CoT  (Diff = With - Without)")
print("=" * 100)
print(compact.to_string())
compact.to_csv(os.path.join(RESULTS_DIR, "cot_comparison_with_vs_without.csv"))

if missing:
    print(f"\nNo results for {len(missing)} run(s): " + "; ".join(missing))
print(f"\nSaved all tables and raw predictions in {RESULTS_DIR}")
print("\nColumns:")
print("  N / Invalid             -- questions scored / answers that were not Yes or No")
print("  Accuracy (%)            -- overall exact-match accuracy")
print("  Precision/Recall/F1 (macro)    -- unweighted average across Yes/No classes")
print("  Precision/Recall/F1 (weighted) -- weighted by class support")
print("  Yes-class / No-class P/R/F1    -- per-class precision/recall/F1")
# A standalone report to download or share, including every full summary table.
report_path = os.path.join(RESULTS_DIR, "FINAL_MODEL_SUMMARY.txt")
with open(report_path, "w", encoding="utf-8") as f:
    f.write("FINAL REPORT: 30/30 MODELS COMPLETED — FULL TEST SETS\n")
    f.write("CoT labels describe training. All models use the same image/question prompt.\n")
    f.write(f"Data root: {DATA_ROOT} | Modality: CT | 4-bit weights / FP16 compute\n")
    f.write(f"Max new tokens: {MAX_NEW_TOKENS} | Image size cap: {MAX_IMAGE_SIDE}\n")
    f.write(f"Invalid prediction policy: {INVALID_PRED_POLICY}\n\n")
    f.write(summary_df.to_string(index=False) + "\n")
    for dataset in DATASETS:
        f.write(f"\nFULL SUMMARY TABLE — {dataset} — 10/10 models\n")
        f.write(summary_df[summary_df["Dataset"] == dataset].to_string(index=False) + "\n")
    f.write("\nWITH vs WITHOUT CoT (Diff = With - Without)\n")
    f.write(compact.to_string() + "\n")
    f.write("\nP/R/F1 = precision/recall/F1; macro = class average; weighted = support-weighted.\n")
print(f"\nComplete shareable report: {report_path}")



FINAL REPORT: 30/30 MODELS COMPLETED — FULL TEST SETS
Input: image + shared question prompt | Output: Yes/No | CoT labels: training variant
Precision: 4-bit weights / FP16 compute | Max new tokens: 8
Image size cap: None | Invalid prediction policy: wrong

ALL DATASETS — ALL 30 MODELS — ALL METRICS
  Dataset    Model Name     CoT   N  Invalid  Accuracy (%)  Precision (macro)  Recall (macro)  F1 (macro)  Precision (weighted)  Recall (weighted)  F1 (weighted)  Yes-class P/R/F1   No-class P/R/F1
BrainFace    Lingshu-7B    With  12        0        100.00             100.00          100.00      100.00                100.00             100.00         100.00 100.0/100.0/100.0 100.0/100.0/100.0
BrainFace    Lingshu-7B Without  12        0        100.00             100.00          100.00      100.00                100.00             100.00         100.00 100.0/100.0/100.0 100.0/100.0/100.0
BrainFace   MedGemma-4B    With  12        0         91.67              91.67           92.86       91.61